In [ ]:
import msprime, tskit
import numpy as np
import gaiapy as gp
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from tqdm import tqdm
from time import time
import json


In [ ]:
#function that takes the tree sequence and returns a array with the columns 
# node id | wether its a sample or ancestor | x coordinate | y coordinate 
def locations(ts):
    nodes = ts.nodes()
    locs_array = []
    for node in nodes:
        if node.individual != -1:
            ind = ts.individual(node.individual)
            x = ind.location[0]
            y = ind.location[1]
            is_sample = node.is_sample()
            locs_array.append(node.id)
            locs_array.append(is_sample)
            locs_array.append(x)
            locs_array.append(y)
        
    locs_array = np.array(locs_array)
    locs = locs_array.reshape(-1, 4)
    return locs


#function that takes the tree sequence and finds the unary nodes in the tree sequence
# returns a list of all the unary indices by node id 
def findUnary(ts):
    unary_nodes = np.zeros(ts.num_nodes) # binary vector specifying if a node is unary or not anywhere on the tree sequence
    for tree in ts.trees():
        num_children = tree.num_children_array[:ts.num_nodes]
        is_unary = num_children == 1
        for i, condition in enumerate(is_unary):
            if is_unary[i] == True:
                unary_nodes[i] = 1
    mask = unary_nodes == 1
    #mask, unary_nodes[mask]
    unary_list = np.where(mask)
    unary_indices = unary_list[0]
    return unary_indices

def node_spans(ts, include_missing=False):
    """
    Returns the array of "node spans", i.e., the `j`th entry gives
    the total span over which node `j` is in the tree sequence.
    Sample nodes that are isolated are "missing data"; inclusion
    of these spans are controlled by `include_missing`. (If
    `include_missing` is `True` then the span of each sample is
    always equal to the sequence length.)

    :param bool include_missing: Whether to include spans of nodes
        on which they have missing data.
    """
    child_spans = np.bincount(
        ts.edges_child,
        weights=ts.edges_right - ts.edges_left,
        minlength=ts.num_nodes,
    )
    for t in ts.trees():
        span = t.span
        for r in t.roots:
            # do this check to exempt 'missing data'
            if include_missing or (t.num_children(r) > 0):
                child_spans[r] += span
    return child_spans



# #here ts is the unsimplified tree
# def get_node_stats(ts):
#     #nodes = list(ts.nodes())
#     nodes = np.array(list(ts.nodes()))

#     slim_ids = np.zeros(ts.num_nodes)
#     for n in ts.nodes():

#         slim_id = n.metadata["slim_id"]
#         assert slim_id not in slim_ids
        
#         # print("made past  slim_id = n.metadata[slim_id]")
#         slim_ids[n.id] = slim_id

#     # Find the samples
#     #is_sample = np.asarray(np.isin(nodes, ts.samples()), dtype=int)
#     is_sample = np.asarray(np.isin(slim_ids, ts.samples()), dtype=int)
#     # Find all the other things (this requires checking tree by tree)
#     tree = ts.first()
#     start = tree.interval[0] 
#     end = ts.sequence_length
#     num_children = np.zeros(nodes.shape[0])
#     num_parents = np.zeros(nodes.shape[0])
#     distinct_children, distinct_parents = list(), list()
#     is_root = np.zeros(nodes.shape[0])
#     #for i,node in tqdm(enumerate(nodes)):
#     for i, slim_id in tqdm(enumerate(slim_ids)): 
#         tree.seek(start)
#         children = list()
#         parents = list()
#         is_root[i] = tree.is_root(i)
#         node_children = list(tree.children(i))

#         children.extend(node_children)
#         parents.append(tree.parent(i))

#         w = (None, tree.interval[0])
#         while w[1] < end and tree.next():
#             w = (w[1], min(tree.interval[1], end))
#             is_root[i] = tree.is_root(i)
#             node_children = list(tree.children(i))

#             children.extend(node_children)
#             parents.append(tree.parent(i))

#         distinct_parents.append(np.unique(parents))
#         distinct_children.append(np.unique(children))
#         num_children[i] = np.unique(children).shape[0]
#         num_parents[i] = np.unique(parents).shape[0]
    
#     data_dict = {
#         'slim_id': slim_ids,
#         'node_id': np.arange(slim_ids.shape[0]),
#         'num_children': num_children,
#         'distinct_children': distinct_children,
#         'distinct_parents': distinct_parents,
#         'num_parents': num_parents,
#         'is_sample': is_sample,
#         'is_root': is_root,
#     }
#     return pd.DataFrame(data_dict)



In [ ]:
ts = tskit.load("tree-S0.2-R0.trees")


In [ ]:
def get_node_stats_fast(ts):
    node_ids = np.array([n.id for n in ts.nodes()])
    is_sample = np.isin(node_ids, ts.samples()).astype(int)
    
    children_set = {}
    for nid in node_ids:
        children_set[nid] = set()

    parents_set = {}
    for nid in node_ids:
        parents_set[nid] = set()

    is_root = {}
    for nid in node_ids:
        is_root[nid] = 0

    for tree in tqdm(ts.trees()): 
        for nid in node_ids:
            if tree.is_root(nid):
                is_root[nid] = 1
            children_set[nid].update(tree.children(nid))
            p = tree.parent(nid)
            if p != -1:
                parents_set[nid].add(p)

    distinct_children = []
    for nid in node_ids:
        as_list = list(children_set[nid])
        as_array = np.array(as_list)
        distinct_children.append(as_array)

    distinct_parents = []
    for nid in node_ids:
        as_list = list(parents_set[nid])
        as_array = np.array(as_list)
        distinct_parents.append(as_array)

    data_dict = pd.DataFrame({
        'id': node_ids,
        'num_children': [len(children_set[nid]) for nid in node_ids],
        'distinct_children': distinct_children,
        'distinct_parents': distinct_parents,
        'num_parents': [len(parents_set[nid])  for nid in node_ids],
        'is_sample': is_sample,
        'is_root': np.array([is_root[nid] for nid in node_ids])
    })

    return data_dict

In [ ]:
#here ts is the unsimplified tree
def get_node_stats(ts):
    #nodes = list(ts.nodes())
    nodes = np.array(list(ts.nodes()))
    node_ids = np.array([n.id for n in nodes])
    # Find the samples
    #is_sample = np.asarray(np.isin(nodes, ts.samples()), dtype=int)
    is_sample = np.asarray(np.isin(node_ids, ts.samples()), dtype=int)
    # Find all the other things (this requires checking tree by tree)
    tree = ts.first()
    start = tree.interval[0] 
    end = ts.sequence_length
    num_children = np.zeros(nodes.shape[0])
    num_parents = np.zeros(nodes.shape[0])
    distinct_children, distinct_parents, distinct_populations = list(), list(), list()
    is_root = np.zeros(nodes.shape[0])
    #for i,node in tqdm(enumerate(nodes)):
    for i, node_id in tqdm(enumerate(node_ids)): 
        tree.seek(start)
        children = list()
        parents = list()
        is_root[i] = tree.is_root(node_id)
        node_children = list(tree.children(node_id))

        children.extend(node_children)
        parents.append(tree.parent(node_id))

        w = (None, tree.interval[0])
        while w[1] < end and tree.next():
            w = (w[1], min(tree.interval[1], end))
            is_root[i] = tree.is_root(node_id)
            node_children = list(tree.children(node_id))

            children.extend(node_children)
            parents.append(tree.parent(node_id))

        distinct_parents.append(np.unique(parents))
        distinct_children.append(np.unique(children))
        num_children[i] = np.unique(children).shape[0]
        num_parents[i] = np.unique(parents).shape[0]
    
    data_dict = {
        'id': node_ids,
        'num_children': num_children,
        'distinct_children': distinct_children,
        'distinct_parents': distinct_parents,
        'num_parents': num_parents,
        'is_sample': is_sample,
        'is_root': is_root,
    }
    return pd.DataFrame(data_dict)

In [ ]:
#ts2 = msprime.sim_ancestry(1e5, sequence_length=1e8, recombination_rate=1e-8, population_size=1e5, coalescing_segments_only=False, random_seed=1)
# Remove the isolated unary spans from our tree sequence
#t2 = remove_isolated_unary(ts2)
# s2 = t2.simplify()
# e2 = s2.extend_haplotypes()


In [ ]:
def get_span_stats(ts, ets):
    
    node_map = {}
    added_span = np.zeros(ets.num_nodes)
    wrong_added_span = np.zeros(ets.num_nodes)
    for n in ts.nodes():

        slim_id = n.metadata["slim_id"]
        assert slim_id not in node_map
        node_map[slim_id] = n.id

    for interval, t, et in ts.coiterate(ets):
        interval_length = interval[1] - interval[0]
        t_nodes = list(t.nodes())
        #et_nodes = list(et.nodes())
        for n in et.nodes():
            # print("et nodes", et_nodes)
            if et.num_children(n) == 1:
                added_span[n] += interval_length
            #on = node_map[n]
            # for x in on:
            node = ets.node(n)
            on = node_map[node.metadata["slim_id"]]
            if on not in t_nodes:
                assert et.num_children(n) == 1, print(interval, n, et.num_children(n), et.time(n))
                wrong_added_span[n] += interval_length
                #print("added ", interval_length, " to wrong_added_span")
    # print("added_span", added_span)
    # print("wrong_added_span", wrong_added_span) 
    # # assert not np.array_equal(added_span, wrong_added_span)
    # span_df = pd.DataFrame({
    #     'added_span': added_span,
    #     'wrong_span': wrong_added_span
    # })
    # span_df.to_csv(f"testing_Span.csv",
    #                 mode='a', header=True, index=False)
    return added_span, wrong_added_span


# try deletign the time part - parse on node id 

In [ ]:
ts = tskit.load("tree-S0.2-R0.trees")
# ts.tables.nodes
# sts = ts.simplify(filter_nodes=False)
# ets = sts.extend_haplotypes()


big_fast_stats = get_node_stats_fast(ts)
big_fast_stats.to_csv("fast_node_tst.csv", mode='a', index=False)


#check out slim_id ...


In [ ]:
bob = msprime.sim_ancestry(3, recombination_rate=0.3, sequence_length=3, record_full_arg=True, random_seed=42, record_provenance=True)
# Example: store age and location for each individual
tables = bob.dump_tables()

node_schema = tskit.MetadataSchema({
    "codec": "json",
    "type": "object",
    "properties": {
        "slim_id": {"type": "integer"}
    }
})
tables.nodes.metadata_schema = node_schema

for node_id, node in enumerate(tables.nodes):
    tables.nodes[node_id] = node.replace(metadata={"slim_id": node_id})


bobby = tables.tree_sequence()

# tested.tables.nodes

sbobby = bobby.simplify(filter_nodes=False)
ebobby = sbobby.extend_haplotypes()


# bob_added, bob_wrong = testing_get_span_stats(bobby, ebobby)
# other, flarp = new_get_span_stats(bobby, ebobby)
# print("added_span", bob_added)
# print("wrond_added_span", bob_wrong)

In [ ]:
bob = msprime.sim_ancestry(3, recombination_rate=0.3, sequence_length=3, record_full_arg=True, random_seed=42, record_provenance=True)
# Example: store age and location for each individual
tables = bob.dump_tables()

node_schema = tskit.MetadataSchema({
    "codec": "json",
    "type": "object",
    "properties": {
        "slim_id": {"type": "integer"}
    }
})
tables.nodes.metadata_schema = node_schema

for node_id, node in enumerate(tables.nodes):
    tables.nodes[node_id] = node.replace(metadata={"slim_id": node_id})


bobby = tables.tree_sequence()

# tested.tables.nodes

sbobby = bobby.simplify(filter_nodes=False)
ebobby = sbobby.extend_haplotypes()


bob_added, bob_wrong = testing_get_span_stats(bobby, ebobby)
other, flarp = new_get_span_stats(bobby, ebobby)
print("added_span", bob_added)
print("wrond_added_span", bob_wrong)

In [ ]:
test = msprime.sim_ancestry(
    samples=100,            # 6 diploid individuals = 12 haploid chromosomes
    population_size=1000,
    sequence_length=10_000,
    recombination_rate=1e-6,   # low recomb → likely a single tree
    random_seed=42,
    record_full_arg=True,
    #coalescing_segments_only=False,
    record_provenance=True
)

# Example: store age and location for each individual
tables = test.dump_tables()

node_schema = tskit.MetadataSchema({
    "codec": "json",
    "type": "object",
    "properties": {
        "slim_id": {"type": "integer"}
    }
})
tables.nodes.metadata_schema = node_schema

for node_id, node in enumerate(tables.nodes):
    tables.nodes[node_id] = node.replace(metadata={"slim_id": node_id})

# for node_id, node in enumerate(tables.nodes):
#     # metadata_dict = {
#     #     "slim_id": node_id, # arbitrary example
#     # }
#     metadata_dict = [{"slim_id": node_id}]
#     # Encode metadata as JSON (tskit stores bytes)
#     tables.nodes[node_id] = node.replace(
#         metadata=json.dumps(metadata_dict).encode("utf-8")
#     )

tested = tables.tree_sequence()

# tested.tables.nodes
tested = remove_unary_nodes(tested)

stested = tested.simplify(filter_nodes=False)
etested = stested.extend_haplotypes()


test_stats, etest_stats = new_get_span_stats(tested, etested)

# these here !! are not equal !! it worked here!!!


# test.draw_svg()

In [ ]:
test = msprime.sim_ancestry(
    samples=100,            # 6 diploid individuals = 12 haploid chromosomes
    population_size=1000,
    sequence_length=10_000,
    recombination_rate=1e-6,   # low recomb → likely a single tree
    random_seed=42,
    record_full_arg=True,
    #coalescing_segments_only=False,
    record_provenance=True
)

# Example: store age and location for each individual
tables = test.dump_tables()

node_schema = tskit.MetadataSchema({
    "codec": "json",
    "type": "object",
    "properties": {
        "slim_id": {"type": "integer"}
    }
})
tables.nodes.metadata_schema = node_schema

for node_id, node in enumerate(tables.nodes):
    tables.nodes[node_id] = node.replace(metadata={"slim_id": node_id})

# for node_id, node in enumerate(tables.nodes):
#     # metadata_dict = {
#     #     "slim_id": node_id, # arbitrary example
#     # }
#     metadata_dict = [{"slim_id": node_id}]
#     # Encode metadata as JSON (tskit stores bytes)
#     tables.nodes[node_id] = node.replace(
#         metadata=json.dumps(metadata_dict).encode("utf-8")
#     )

# tested = tables.tree_sequence()

# # tested.tables.nodes
# tested = remove_unary_nodes(tested)

# stested = tested.simplify(filter_nodes=False)
# etested = stested.extend_haplotypes()


# test_stats, etest_stats = new_get_span_stats(tested, etested)

# these here !! are not equal !! it worked here!!!


# test.draw_svg()

In [ ]:
def example2():
        node_times = {
            0: 0,
            1: 0,
            2: 0,
            3: 0,
            4: 0,
            5: 0,
            6: 0,
            7: 0,
            8: 0,
            9: 0,
            10: 1,
            11: 2,
            12: 3,
            13: 4,
            14: 5,
            15: 6,
            16: 7,
            17: 8,
            18: 9,
            19: 10,
            20: 11,
            21: 12,
        }
        # (p,c,l,r)
        edges = [
            (10, 2, 0, 9),
            (10, 5, 0, 9),
            (11, 0, 0, 9),
            (11, 7, 0, 9),
            (12, 3, 3, 9),
            (12, 9, 3, 9),
            (13, 4, 0, 9),
            (13, 11, 0, 9),
            (14, 6, 0, 9),
            (14, 10, 0, 9),
            (15, 9, 0, 3),
            (15, 13, 0, 3),
            (16, 1, 0, 6),
            (16, 3, 0, 3),
            (16, 12, 3, 6),
            (17, 1, 6, 9),
            (17, 13, 6, 9),
            (18, 13, 3, 6),
            (18, 14, 0, 9),
            (18, 15, 0, 3),
            (18, 17, 6, 9),
            (19, 8, 0, 9),
            (19, 12, 6, 9),
            (19, 16, 0, 6),
            (20, 18, 0, 3),
            (20, 18, 6, 9),
            (20, 19, 0, 3),
            (20, 19, 6, 9),
            (21, 18, 3, 6),
            (21, 19, 3, 6),
        ]
        extended_edges = [
            (10, 2, 0.0, 9.0),
            (10, 5, 0.0, 9.0),
            (11, 0, 0.0, 9.0),
            (11, 7, 0.0, 9.0),
            (12, 3, 0.0, 9.0),
            (12, 9, 3.0, 9.0),
            (13, 4, 0.0, 9.0),
            (13, 11, 0.0, 9.0),
            (14, 6, 0.0, 9.0),
            (14, 10, 0.0, 9.0),
            (15, 9, 0.0, 3.0),
            (15, 13, 0.0, 9.0),
            (16, 1, 0.0, 6.0),
            (16, 12, 0.0, 9.0),
            (17, 1, 6.0, 9.0),
            (17, 15, 0.0, 9.0),
            (18, 14, 0.0, 9.0),
            (18, 17, 0.0, 9.0),
            (19, 8, 0.0, 9.0),
            (19, 16, 0.0, 9.0),
            (20, 18, 0.0, 3.0),
            (20, 18, 6.0, 9.0),
            (20, 19, 0.0, 3.0),
            (20, 19, 6.0, 9.0),
            (21, 18, 3.0, 6.0),
            (21, 19, 3.0, 6.0),
        ]
        samples = list(np.arange(10))
        tables = tskit.TableCollection(sequence_length=9)
        for (
            n,
            t,
        ) in node_times.items():
            flags = tskit.NODE_IS_SAMPLE if n in samples else 0
            tables.nodes.add_row(time=t, flags=flags)
        for p, c, l, r in edges:
            tables.edges.add_row(parent=p, child=c, left=l, right=r)
        ts = tables.tree_sequence()
        tables.edges.clear()
        for p, c, l, r in extended_edges:
            tables.edges.add_row(parent=p, child=c, left=l, right=r)
        ets = tables.tree_sequence()
        assert ts.num_edges == 30
        assert ets.num_edges == 26
        return ts, ets

samples2 = np.array([
    [0, 1.5, 2.0],  # node 0 at coordinates (1.5, 2.0)
    [1, 4.2, 3.1],  # node 1 at coordinates (4.2, 3.1) 
    [2, 6.6, 5.5],  # node 2 at coordinates (6.7, 5.5)
    [3, 6.8, 5.5],  # node 2 at coordinates (6.7, 5.5) 
    [4, 9.0, 10.5],  # node 2 at coordinates (6.7, 5.5) 
    [5, 11.3, 2.5],  # node 2 at coordinates (6.7, 5.5) 
    [6, 11.2, 5.4],  # node 2 at coordinates (6.7, 5.5) 
    [7, 12.0, 6.9],  # node 2 at coordinates (6.7, 5.5) 
    [8, 11.1, 4.7],  # node 2 at coordinates (6.7, 5.5) 
    [9, 1.6, 2.0],  # node 0 at coordinates (1.5, 2.0) 
])

ancestors2 = np.array([
    [10, 2, 2.1],  # node 0 at coordinates (1.5, 2.0)
    [11, 3.6, 5.2],  # node 0 at coordinates (1.5, 2.0)
    [12, 4.1, 7.8],  # node 0 at coordinates (1.5, 2.0)
    [13, 7.8, 8.9],  # node 0 at coordinates (1.5, 2.0)
    [14, 5.6, 2.0],  # node 0 at coordinates (1.5, 2.0)
    [15, 9.7, 3.1],  # node 0 at coordinates (1.5, 2.0)
    [16, 9.8, 11.1],  # node 0 at coordinates (1.5, 2.0)
    [17, 10.1, 11.6],  # node 0 at coordinates (1.5, 2.0)
    [18, 11.2, 2.1],  # node 0 at coordinates (1.5, 2.0)
    [19, 14.1, 4.0],  # node 0 at coordinates (1.5, 2.0)
    [20, 1.2, 8.9],  # node 0 at coordinates (1.5, 2.0)
    [21, 4.3, 8.7],  # node 0 at coordinates (1.5, 2.0)
])

everything2 = np.array([
    [0, 1.5, 2.0],  # node 0 at coordinates (1.5, 2.0)
    [1, 4.2, 3.1],  # node 1 at coordinates (4.2, 3.1) 
    [2, 6.6, 5.5],  # node 2 at coordinates (6.7, 5.5)
    [3, 6.8, 5.5],  # node 2 at coordinates (6.7, 5.5) 
    [4, 9.0, 10.5],  # node 2 at coordinates (6.7, 5.5) 
    [5, 11.3, 2.5],  # node 2 at coordinates (6.7, 5.5) 
    [6, 11.2, 5.4],  # node 2 at coordinates (6.7, 5.5) 
    [7, 12.0, 6.9],  # node 2 at coordinates (6.7, 5.5) 
    [8, 11.1, 4.7],  # node 2 at coordinates (6.7, 5.5) 
    [9, 1.6, 2.0],  # node 0 at coordinates (1.5, 2.0) 
    [10, 2, 2.1],  # node 0 at coordinates (1.5, 2.0)
    [11, 3.6, 5.2],  # node 0 at coordinates (1.5, 2.0)
    [12, 4.1, 7.8],  # node 0 at coordinates (1.5, 2.0)
    [13, 7.8, 8.9],  # node 0 at coordinates (1.5, 2.0)
    [14, 5.6, 2.0],  # node 0 at coordinates (1.5, 2.0)
    [15, 9.7, 3.1],  # node 0 at coordinates (1.5, 2.0)
    [16, 9.8, 11.1],  # node 0 at coordinates (1.5, 2.0)
    [17, 10.1, 11.6],  # node 0 at coordinates (1.5, 2.0)
    [18, 11.2, 2.1],  # node 0 at coordinates (1.5, 2.0)
    [19, 14.1, 4.0],  # node 0 at coordinates (1.5, 2.0)
    [20, 1.2, 8.9],  # node 0 at coordinates (1.5, 2.0)
    [21, 4.3, 8.7],  # node 0 at coordinates (1.5, 2.0)
])

t2, et2 = example2()

get_span_stats(t2, et2)


#small_getAccOut(t2, samples2, ancestors2, everything2)
# total_added_span, wrongly_added_span = get_span_stats(t2, et2)
# total_added_span
# print(f"Out of a total of {total_added_span} added edge span, "
#       f"we have wrongly added {wrongly_added_span} span, "
#       f"a proportion of {wrongly_added_span / total_added_span}.")


In [ ]:
old_get_span_stats(t2, et2)

In [ ]:
def example1():
        node_times = {
            0: 0,
            1: 0,
            2: 0,
            3: 0,
            4: 1,
            5: 1,
            6: 4,
            7: 6,
            8: 10,
            9: 4,
            10: 12,
            11: 8,
            12: 8,
            13: 15,
        }
        # (p,c,l,r)
        edges = [
            (4, 0, 0, 9),
            (4, 1, 0, 9),
            (5, 2, 0, 6),
            (5, 3, 0, 9),
            (6, 4, 0, 3),
            (9, 5, 0, 3),
            (7, 4, 3, 6),
            (11, 7, 3, 6),
            (12, 5, 3, 6),
            (8, 2, 6, 9),
            (8, 4, 6, 9),
            (8, 6, 0, 3),
            (10, 5, 6, 9),
            (10, 8, 0, 3),
            (10, 8, 6, 9),
            (10, 9, 0, 3),
            (10, 11, 3, 6),
            (10, 12, 3, 6),
            (13, 10, 3, 6),
        ]
        extended_edges = [
            (4, 0, 0.0, 9.0),
            (4, 1, 0.0, 9.0),
            (5, 2, 0.0, 6.0),
            (5, 3, 0.0, 9.0),
            (6, 4, 0.0, 9.0),
            (9, 5, 0.0, 9.0),
            (7, 6, 0.0, 9.0),
            (11, 7, 0.0, 9.0),
            (12, 9, 0.0, 9.0),
            (8, 2, 6.0, 9.0),
            (8, 11, 0.0, 9.0),
            (10, 8, 0.0, 9.0),
            (10, 12, 0.0, 9.0),
            (13, 10, 3.0, 6.0),
        ]
        samples = list(np.arange(4))
        tables = tskit.TableCollection(sequence_length=9)
        for (
            n,
            t,
        ) in node_times.items():
            flags = tskit.NODE_IS_SAMPLE if n in samples else 0
            tables.nodes.add_row(time=t, flags=flags)
        for p, c, l, r in edges:
            tables.edges.add_row(parent=p, child=c, left=l, right=r)
        ts = tables.tree_sequence()
        tables.edges.clear()
        for p, c, l, r in extended_edges:
            tables.edges.add_row(parent=p, child=c, left=l, right=r)
        ets = tables.tree_sequence()
        assert ts.num_edges == 19
        assert ets.num_edges == 14
        return ts, ets

t1, et1 = example1()


In [ ]:
def attempt1():
    node_times = (0, 0, 0, 1, 2, 3)
    samples = (0, 1, 2)
    # (p, c, l, r)
    extended_edges = [
        (3, 0, 0, 4),
        (3, 1, 0, 3), 
        (4, 1, 3, 4),
        (4, 2, 0, 4), 
        (5, 3, 0, 4), 
        (5, 4, 0, 4),
    ]
    edges = [
        (3, 0, 0, 3),
        (3, 1, 0, 3), 
        (4, 1, 3, 4),
        (4, 2, 3, 4), 
        (5, 0, 3, 4),
        (5, 2, 0, 3), 
        (5, 3, 0, 3), 
        (5, 4, 3, 4),
    ]
    tables = tskit.TableCollection(sequence_length=4)
    tables.sort()
    for n, t in enumerate(node_times):
        flags = tskit.NODE_IS_SAMPLE if n in samples else 0
        tables.nodes.add_row(time=t, flags=flags)
    for p, c, l, r in edges:
        tables.edges.add_row(parent=p, child=c, left=l, right=r)
    ts = tables.tree_sequence()
    tables.edges.clear()
    for p, c, l, r in extended_edges:
        tables.edges.add_row(parent=p, child=c, left=l, right=r)
    ets = tables.tree_sequence()
    assert ts.num_edges == 8
    assert ets.num_edges == 6
    return ts, ets

t5, et5 = attempt1()
